In [ ]:
# Install packages
!pip install langgraph langchain-openai langchain -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.6 MB/s eta 0:00:00


In [ ]:
# Imports and API Key
import os
from getpass import getpass
from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage

# Load OpenAI API Key
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

print("API Key loaded.")

API Key loaded.


In [ ]:
# Define Defensive Tools

@tool
def enrich_alert(alert_id: str) -> str:
    """Enrich a security alert with threat intelligence and context."""
    return (
        f"Alert {alert_id} enriched.\n"
        f"- Detected C2 beaconing behavior from IP 192.168.1.45 to external host.\n"
        f"- MITRE ATT&CK: T1071.001 (Application Layer Protocol)\n"
        f"- Confidence: High\n"
        f"- Recommended action: Notify SOC team for investigation."
    )

@tool
def send_notification(message: str) -> str:
    """Send a notification to the security team (requires human approval)."""
    return f"Notification sent to SOC channel:\n{message}"

print("Tools defined.")

Tools defined.


In [ ]:
# Create LLM and Specialized Agents

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=600
)

# Each agent gets only the tool it needs
enrichment_agent = create_agent(llm, [enrich_alert])
notification_agent = create_agent(llm, [send_notification])

print("Agents created successfully.")

Agents created successfully.


In [ ]:
# Run Defensive C2 Anomaly Agent with Human-in-the-Loop

print("=== Defensive C2 Anomaly Detection Agent ===\n")

# Simulated incoming high-severity alert
alert = {
    "alert_id": "ALERT-78432",
    "host": "WEB-PROD-07",
    "description": "High severity alert: Unusual outbound traffic detected"
}

print(f"Incoming Alert: {alert['alert_id']} on {alert['host']}")
print(f"Description: {alert['description']}\n")

def get_content(msg):
    """Safely extract text from LLM response."""
    content = msg.content
    if isinstance(content, dict) and 'text' in content:
        return content['text']
    if isinstance(content, list):
        return "".join([b.get('text', str(b)) if isinstance(b, dict) else str(b) for b in content])
    return str(content)

# Step 1: Enrich the alert
print(">>> Step 1: Enriching Alert")
result = enrichment_agent.invoke({
    "messages": [
        SystemMessage(content="You must only use the tools provided to you."),
        HumanMessage(content=f"Enrich alert {alert['alert_id']} for host {alert['host']}")
    ]
})
enrichment_output = get_content(result["messages"][-1])
print(enrichment_output + "\n")

# Human Approval Gate
approval = input("Do you want to notify the SOC team? (yes/no): ").strip().lower()

if approval != "yes":
    print("\nNotification skipped by human operator. No action taken.")
else:
    # Step 2: Send notification (only if approved)
    print("\n>>> Step 2: Sending Notification to SOC")
    result = notification_agent.invoke({
        "messages": [
            SystemMessage(content="You must only use the tools provided to you."),
            HumanMessage(content=f"Send notification with the following details:\n{enrichment_output}")
        ]
    })
    print(get_content(result["messages"][-1]))

print("\n=== Defensive Agent Run Complete ===")

=== Defensive C2 Anomaly Detection Agent ===

Incoming Alert: ALERT-78432 on WEB-PROD-07
Description: High severity alert: Unusual outbound traffic detected

>>> Step 1: Enriching Alert
The security alert **ALERT-78432** for host **WEB-PROD-07** has been successfully enriched with the following threat intelligence and context:

*   **Detection:** C2 beaconing behavior from internal IP `192.168.1.45` to an external host.
*   **MITRE ATT&CK Technique:** [T1071.001 (Application Layer Protocol: Web Protocols)](https://attack.mitre.org/techniques/T1071/001/)
*   **Confidence Level:** High
*   **Recommended Action:** Immediately notify the SOC (Security Operations Center) team for further investigation and potential containment of the host.

Do you want to notify the SOC team? (yes/no): yes

>>> Step 2: Sending Notification to SOC
The notification has been successfully sent to the security team with the details provided.

=== Defensive Agent Run Complete ===
